# Step 10: Zusammenfassung und Ausfuehrung

Dieses Notebook ist die zusammenfassende Projektsteuerung. Es enthaelt keine duplizierte Trainings-, Evaluations- oder Video-Blur-Logik mehr, sondern setzt Parameter und ruft die Python-Skripte aus `face_model_lab/` auf.


## Skripte statt Notebook-Dopplungen

Verwendete Skripte:

- [`step02_train_ultralytics_detector.py`](step02_train_ultralytics_detector.py): YOLO/Ultralytics-Training und WIDER-FACE-zu-YOLO-Konvertierung.
- [`step03_train_torchvision_detector.py`](step03_train_torchvision_detector.py): Faster R-CNN, RetinaNet und FCOS mit gemeinsamer Torchvision-Trainingslogik.
- [`step06_evaluate_models.py`](step06_evaluate_models.py): Vergleichsevaluation, CSV/JSON-Ergebnisse und Vergleichsgrafiken.
- [`step08_blur_video.py`](step08_blur_video.py): Videoanonymisierung mit dem gewaehlten Modell.

`step04_train_fasterrcnn.py` und `step05_train_yolo_legacy.py` bleiben als alte/spezialisierte Varianten erhalten, werden hier aber nicht aufgerufen, damit keine doppelte Logik entsteht.


## Parameter

Die hier gesetzten Werte entsprechen dem angeforderten Lauf: Reduktion 20, Batch Size 2. Auf ROCm/AMD erscheint die GPU in PyTorch und Ultralytics ueber die CUDA-Schnittstelle; `cuda` bzw. `device=0` ist deshalb korrekt fuer die AMD Radeon PRO W7800.


In [1]:
from pathlib import Path
import shlex
import subprocess

ROOT = Path('/home/clemi/projekte/MIM')
LAB = ROOT / 'face_model_lab'
PYTHON = Path('/home/clemi/.venvs/MIM/bin/python')

REDUCTION = 20
BATCH_SIZE = 2
EPOCHS = 1
IMGSZ = 768
WORKERS = 0
EVAL_LIMIT = 300
IOU = 0.4
CONF = 0.25

# Standardmaessig werden beim Ausfuehren des Notebooks keine langen Trainings neu gestartet.
# Auf True setzen, wenn die Schritte direkt aus diesem Notebook erneut laufen sollen.
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_VIDEO_SMOKE = False

YOLO_MODEL = ROOT / 'trained_models/yolo_yolov8m_widerface_rocm_bs2_red20_ep1.pt'
FASTER_MODEL = ROOT / 'trained_models/fasterrcnn_resnet50_fpn_rocm_bs2_red20_ep1.pth'
RETINA_MODEL = ROOT / 'trained_models/retinanet_resnet50_fpn_rocm_bs2_red20_ep1.pth'
FCOS_MODEL = ROOT / 'trained_models/fcos_resnet50_fpn_rocm_bs2_red20_ep1.pth'
VIDEO_IN = ROOT / 'Videos/Testmaterial Gesichter.mp4'
VIDEO_OUT = ROOT / 'Videos/lab_outputs/testmaterial_red20_yolo_smoke_blur.mp4'
VIDEO_PREVIEW_OUT = ROOT / 'Videos/lab_outputs/testmaterial_red20_yolo_preview_ids.mp4'
VIDEO_SELECTED_OUT = ROOT / 'Videos/lab_outputs/testmaterial_red20_yolo_selected_blur.mp4'

# Selektives Blurring: nach Preview-Video IDs/Raenge/Regionen eintragen.
TARGET_TRACK_IDS = ''
TARGET_RANKS = ''
TARGET_REGIONS = ''


In [2]:
def run(cmd, cwd=ROOT):
    printable = ' '.join(shlex.quote(str(part)) for part in cmd)
    print('$', printable)
    subprocess.run([str(part) for part in cmd], cwd=str(cwd), check=True)

def show_command(cmd):
    print(' '.join(shlex.quote(str(part)) for part in cmd))


## Training per Skriptaufruf

Diese Zellen rufen ausschliesslich die Skripte auf. Die eigentliche Datenaufbereitung und Trainingslogik bleibt in den Skripten.


In [3]:
yolo_cmd = [
    PYTHON, LAB / 'step02_train_ultralytics_detector.py',
    '--family', 'yolo', '--base', 'yolov8m.pt',
    '--epochs', EPOCHS, '--batch', BATCH_SIZE, '--reduction', REDUCTION,
    '--imgsz', IMGSZ, '--train-limit', 0, '--val-limit', 0, '--workers', WORKERS,
]
show_command(yolo_cmd)
if RUN_TRAINING:
    run(yolo_cmd)


/home/clemi/.venvs/MIM/bin/python /home/clemi/projekte/MIM/face_model_lab/step02_train_ultralytics_detector.py --family yolo --base yolov8m.pt --epochs 1 --batch 2 --reduction 20 --imgsz 768 --train-limit 0 --val-limit 0 --workers 0


In [4]:
torchvision_commands = []
for kind in ['retinanet', 'fcos', 'fasterrcnn']:
    torchvision_commands.append([
        PYTHON, LAB / 'step03_train_torchvision_detector.py',
        '--kind', kind,
        '--epochs', EPOCHS, '--batch', BATCH_SIZE, '--reduction', REDUCTION,
        '--workers', WORKERS, '--save-every', 1,
    ])

for cmd in torchvision_commands:
    show_command(cmd)
    if RUN_TRAINING:
        run(cmd)


/home/clemi/.venvs/MIM/bin/python /home/clemi/projekte/MIM/face_model_lab/step03_train_torchvision_detector.py --kind retinanet --epochs 1 --batch 2 --reduction 20 --workers 0 --save-every 1
/home/clemi/.venvs/MIM/bin/python /home/clemi/projekte/MIM/face_model_lab/step03_train_torchvision_detector.py --kind fcos --epochs 1 --batch 2 --reduction 20 --workers 0 --save-every 1
/home/clemi/.venvs/MIM/bin/python /home/clemi/projekte/MIM/face_model_lab/step03_train_torchvision_detector.py --kind fasterrcnn --epochs 1 --batch 2 --reduction 20 --workers 0 --save-every 1


## Evaluation per Skriptaufruf

Die Evaluation vergleicht die vier trainierten Modelle auf denselben Validierungsbildern und erzeugt CSV/JSON sowie die wichtigsten Plots in `model_results/plots/`.


In [5]:
eval_cmd = [
    PYTHON, LAB / 'step06_evaluate_models.py',
    '--models', YOLO_MODEL, FASTER_MODEL, RETINA_MODEL, FCOS_MODEL,
    '--limit', EVAL_LIMIT, '--iou', IOU, '--conf', CONF, '--imgsz', IMGSZ,
    '--score-thresholds', 0.25, 0.5, 0.7,
    '--include-pretrained-coco', 'fasterrcnn', 'retinanet', 'fcos', 'yolov8m',
]
show_command(eval_cmd)
if RUN_EVALUATION:
    run(eval_cmd)


/home/clemi/.venvs/MIM/bin/python /home/clemi/projekte/MIM/face_model_lab/step06_evaluate_models.py --models /home/clemi/projekte/MIM/trained_models/yolo_yolov8m_widerface_rocm_bs2_red20_ep1.pt /home/clemi/projekte/MIM/trained_models/fasterrcnn_resnet50_fpn_rocm_bs2_red20_ep1.pth /home/clemi/projekte/MIM/trained_models/retinanet_resnet50_fpn_rocm_bs2_red20_ep1.pth /home/clemi/projekte/MIM/trained_models/fcos_resnet50_fpn_rocm_bs2_red20_ep1.pth --limit 300 --iou 0.4 --conf 0.25 --imgsz 768 --score-thresholds 0.25 0.5 0.7 --include-pretrained-coco fasterrcnn retinanet fcos yolov8m


## Video-Blur per Skriptaufruf

Fuer die praktische Anonymisierung wird YOLO verwendet, weil es die Video-Pipeline am direktesten unterstuetzt. Die Smoke-Run-Ausgabe liegt in `Videos/lab_outputs/`.


In [6]:
video_cmd = [
    PYTHON, LAB / 'step08_blur_video.py',
    '--model', YOLO_MODEL,
    '--input', VIDEO_IN,
    '--output', VIDEO_OUT,
    '--conf', CONF,
    '--imgsz', IMGSZ,
    '--max-frames', 120,
    '--frame-skip', 2,
    '--deinterlace', 'smoke',
    '--blur-mode', 'oval',
]
show_command(video_cmd)
if RUN_VIDEO_SMOKE:
    run(video_cmd)


/home/clemi/.venvs/MIM/bin/python /home/clemi/projekte/MIM/face_model_lab/step08_blur_video.py --model /home/clemi/projekte/MIM/trained_models/yolo_yolov8m_widerface_rocm_bs2_red20_ep1.pt --input '/home/clemi/projekte/MIM/Videos/Testmaterial Gesichter.mp4' --output /home/clemi/projekte/MIM/Videos/lab_outputs/testmaterial_red20_yolo_smoke_blur.mp4 --conf 0.25 --imgsz 768 --max-frames 120 --frame-skip 2 --deinterlace smoke --blur-mode oval


## Selektives Video-Blurring

Fuer konkrete Gesichter gibt es einen zweistufigen Ablauf: Erst ein Preview-Video mit Boxen, Track-IDs und Raengen erzeugen, dann im zweiten Lauf gezielt IDs/Raenge/Regionen blurren.


In [7]:
preview_cmd = [
    PYTHON, LAB / 'step08_blur_video.py',
    '--model', YOLO_MODEL,
    '--input', VIDEO_IN,
    '--output', VIDEO_PREVIEW_OUT,
    '--conf', CONF,
    '--imgsz', IMGSZ,
    '--max-frames', 120,
    '--frame-skip', 2,
    '--deinterlace', 'smoke',
    '--mode', 'preview',
]
show_command(preview_cmd)
if RUN_VIDEO_SMOKE:
    run(preview_cmd)

selected_blur_cmd = [
    PYTHON, LAB / 'step08_blur_video.py',
    '--model', YOLO_MODEL,
    '--input', VIDEO_IN,
    '--output', VIDEO_SELECTED_OUT,
    '--conf', CONF,
    '--imgsz', IMGSZ,
    '--max-frames', 120,
    '--frame-skip', 2,
    '--deinterlace', 'smoke',
    '--blur-mode', 'oval',
    '--target-track-ids', TARGET_TRACK_IDS,
    '--target-ranks', TARGET_RANKS,
    '--target-regions', TARGET_REGIONS,
]
show_command(selected_blur_cmd)
if RUN_VIDEO_SMOKE:
    run(selected_blur_cmd)


/home/clemi/.venvs/MIM/bin/python /home/clemi/projekte/MIM/face_model_lab/step08_blur_video.py --model /home/clemi/projekte/MIM/trained_models/yolo_yolov8m_widerface_rocm_bs2_red20_ep1.pt --input '/home/clemi/projekte/MIM/Videos/Testmaterial Gesichter.mp4' --output /home/clemi/projekte/MIM/Videos/lab_outputs/testmaterial_red20_yolo_preview_ids.mp4 --conf 0.25 --imgsz 768 --max-frames 120 --frame-skip 2 --deinterlace smoke --mode preview
/home/clemi/.venvs/MIM/bin/python /home/clemi/projekte/MIM/face_model_lab/step08_blur_video.py --model /home/clemi/projekte/MIM/trained_models/yolo_yolov8m_widerface_rocm_bs2_red20_ep1.pt --input '/home/clemi/projekte/MIM/Videos/Testmaterial Gesichter.mp4' --output /home/clemi/projekte/MIM/Videos/lab_outputs/testmaterial_red20_yolo_selected_blur.mp4 --conf 0.25 --imgsz 768 --max-frames 120 --frame-skip 2 --deinterlace smoke --blur-mode oval --target-track-ids '' --target-ranks '' --target-regions ''


## Ergebnisstand des ausgefuehrten Red20/Batch-2-Laufs

Die Schritte wurden einmal mit den oben gesetzten Parametern ausgefuehrt. Relevante Artefakte:

| Artefakt | Pfad |
|---|---|
| YOLO-Modell | `trained_models/yolo_yolov8m_widerface_rocm_bs2_red20_ep1.pt` |
| Faster R-CNN | `trained_models/fasterrcnn_resnet50_fpn_rocm_bs2_red20_ep1.pth` |
| RetinaNet | `trained_models/retinanet_resnet50_fpn_rocm_bs2_red20_ep1.pth` |
| FCOS | `trained_models/fcos_resnet50_fpn_rocm_bs2_red20_ep1.pth` |
| Evaluation | `model_results/evaluation_20260623_124037.csv` |
| Evaluation JSON | `model_results/evaluation_20260623_124037.json` |
| Video Smoke Output | `Videos/lab_outputs/testmaterial_red20_yolo_smoke_blur.mp4` |
| Preview mit Face-IDs/Raengen | `Videos/lab_outputs/testmaterial_red20_yolo_preview_ids.mp4` |
| Selektiver Blur-Beispiellauf | `Videos/lab_outputs/testmaterial_red20_yolo_selected_blur.mp4` |

Wichtigste Evaluationswerte:

| Modell | Recall | ms/Bild |
|---|---:|---:|
| FCOS red20 ep1 | 0.547 | 36.2 |
| Faster R-CNN red20 ep1 | 0.498 | 51.4 |
| YOLOv8m red20 ep1 | 0.406 | 43.7 |
| RetinaNet red20 ep1 | 0.327 | 35.0 |

Damit ist FCOS in diesem kurzen Red20/1-Epoch-Lauf Recall-Sieger. YOLO bleibt fuer die Video-Anonymisierung besonders wertvoll, weil es Ergebnisse, Confusion Matrix, Inferenz und spaetere Tracking-Anbindung am saubersten in einer Pipeline verbindet.


Selektives Blurring nutzt im ersten Lauf `--mode preview` fuer sichtbare Boxen, Raenge und YOLO-Track-IDs. Im zweiten Lauf koennen konkrete Gesichter mit `--target-track-ids`, `--target-ranks` oder `--target-regions` geblurrt werden.


## Projektstruktur fuer die Praesentation

```text
face_model_lab/              Notebooks und Python-Skripte
annotations/                 COCO-Annotationen fuer Torchvision
datasets/wider_face/         WIDER-FACE-Rohdaten und Splits
trained_models/              trainierte Modelle und YOLO-Runs
model_results/               CSV/JSON-Evaluationen, Lernkurven, Plots
Videos/lab_outputs/          geblurrte Ergebnisvideos
```

Die inhaltliche Vertiefung der Evaluation liegt in `step09_evaluation.ipynb`. Dieses Notebook hier erklaert vor allem, wie die Schritte reproduzierbar aufgerufen werden.
